# Hotel-RAG: chạy Dolphin 3.0 (Llama 3.1 8B) trên Google Colab

Notebook này dựng một Ollama server chạy trên GPU T4 miễn phí của Colab, để backend Hotel-RAG trên máy bạn gọi tới qua Internet.

**Trước khi chạy:** vào menu **Runtime → Change runtime type → chọn `T4 GPU`**, rồi chạy lần lượt từng ô từ trên xuống (Shift + Enter).

⚠️ **Lưu ý:**
- Colab bản miễn phí tự ngắt phiên khi bạn không thao tác một thời gian, và có giới hạn tổng thời gian dùng GPU.
- Điều khoản Colab bản miễn phí hạn chế việc dùng notebook làm dịch vụ web. Chỉ dùng notebook này để **thử nghiệm và demo ngắn**, không dùng để chạy liên tục.
- Mỗi lần mở phiên mới, URL tunnel và khóa đều **thay đổi**, nên phải cập nhật lại `.env`.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cài Ollama

In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Khởi động Ollama server (chạy nền)

In [ ]:
import os, subprocess, time, urllib.request

env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "1h"}
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                               stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama đã chạy")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama chưa chạy, xem file ollama.log")

## 4. Tải model Dolphin 3.0 (khoảng 4.9 GB, mất vài phút)

In [ ]:
MODEL = "dolphin3"   # Dolphin 3.0 Llama 3.1 8B; GPU T4 16 GB chứa trọn model
!ollama pull {MODEL}
!ollama run {MODEL} "Giới thiệu ngắn một khách sạn ven biển ở Đà Nẵng bằng tiếng Việt."

## 5. Proxy xác thực

Ollama không có đăng nhập, còn tunnel ở bước 6 tạo ra một URL công khai. Proxy này chỉ cho request mang đúng khóa (`Authorization: Bearer <khóa>`) đi qua, và chỉ mở 3 API: `/api/chat`, `/api/generate`, `/api/tags`.

In [ ]:
%%writefile auth_proxy.py
# File: colab/auth_proxy.py
# Proxy nhỏ đứng trước Ollama: chỉ cho request có header "Authorization: Bearer <TOKEN>" đi qua.
# Lý do: Ollama gốc không có xác thực, còn tunnel Cloudflare tạo ra URL công khai trên Internet.
# Chỉ dùng thư viện chuẩn Python (Colab có sẵn). Notebook colab/ollama_dolphin_colab.ipynb nạp file này.
import http.server
import threading
import urllib.error
import urllib.request

OLLAMA_UPSTREAM = 'http://127.0.0.1:11434'


def start_auth_proxy(token, port=8000, upstream=OLLAMA_UPSTREAM):
    class Proxy(http.server.BaseHTTPRequestHandler):
        def _reply(self, status, body=b'', content_type='application/json'):
            self.send_response(status)
            self.send_header('Content-Type', content_type)
            self.send_header('Content-Length', str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def _forward(self):
            if self.headers.get('Authorization') != f'Bearer {token}':
                return self._reply(401, b'{"error":"unauthorized"}')
            # Chỉ mở API cần thiết, không mở các API quản trị (pull/delete/create model...)
            if self.path not in ('/api/chat', '/api/generate', '/api/tags'):
                return self._reply(404, b'{"error":"not found"}')

            length = int(self.headers.get('Content-Length') or 0)
            body = self.rfile.read(length) if length else None
            request = urllib.request.Request(
                upstream + self.path, data=body, method=self.command,
                headers={'Content-Type': 'application/json'}
            )
            try:
                with urllib.request.urlopen(request, timeout=300) as res:
                    return self._reply(res.status, res.read())
            except urllib.error.HTTPError as error:
                return self._reply(error.code, error.read())
            except Exception as error:  # Ollama chưa chạy, quá thời gian...
                return self._reply(502, f'{{"error":"{type(error).__name__}"}}'.encode())

        do_GET = _forward
        do_POST = _forward

        def log_message(self, *args):  # tắt log mỗi request cho gọn output Colab
            pass

    server = http.server.ThreadingHTTPServer(('127.0.0.1', port), Proxy)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server

In [ ]:
import secrets
from auth_proxy import start_auth_proxy

OLLAMA_API_KEY = secrets.token_hex(24)   # khóa mới cho mỗi phiên
proxy = start_auth_proxy(OLLAMA_API_KEY, port=8000)
print("Proxy đang chạy ở cổng 8000")

## 6. Mở tunnel Cloudflare (không cần tài khoản)

In [ ]:
import re, subprocess, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                               stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if found:
        PUBLIC_URL = found.group(0)
        break
    time.sleep(1)

if PUBLIC_URL:
    print("Dán các dòng sau vào file .env ở thư mục gốc dự án Hotel-RAG, rồi khởi động lại API:\n")
    print("LLM_PROVIDER=ollama")
    print(f"OLLAMA_MODEL={MODEL}")
    print(f"OLLAMA_URL={PUBLIC_URL}")
    print(f"OLLAMA_API_KEY={OLLAMA_API_KEY}")
    print("OLLAMA_TIMEOUT_MS=90000")
else:
    print("Chưa lấy được URL tunnel, xem file cloudflared.log")

## 7. Kiểm tra qua tunnel (giống cách backend sẽ gọi)

In [ ]:
import json, time, urllib.request

time.sleep(5)   # chờ tunnel sẵn sàng
payload = {
    "model": MODEL, "stream": False,
    "messages": [{"role": "user", "content": "Chào bạn, trả lời 1 câu bằng tiếng Việt."}],
}
req = urllib.request.Request(PUBLIC_URL + "/api/chat", data=json.dumps(payload).encode(), method="POST",
                             headers={"Content-Type": "application/json", "Authorization": f"Bearer {OLLAMA_API_KEY}"})
start = time.time()
with urllib.request.urlopen(req, timeout=120) as res:
    print(json.loads(res.read())["message"]["content"])
print(f"Thời gian: {time.time() - start:.1f}s")

## 8. Giữ phiên chạy

- **Để tab Colab này mở** trong suốt thời gian demo. Đóng tab hoặc phiên hết hạn thì model ngừng hoạt động.
- Khi Colab mất kết nối, backend vẫn trả lời bằng dữ liệu đã lọc (`answerSource: "template"`), nên website không bị lỗi.
- **Dùng xong**, vào **Runtime → Disconnect and delete runtime** để trả GPU và tránh hết hạn mức sử dụng.